# Phase gates: S, T, and their effects

The $S$ and $T$ gates rotate the phase of $|1\rangle$ around the
z-axis without changing measurement probabilities in the
computational basis.  Phase is only visible through interference
or by measuring in a different basis.

In [ ]:
from IPython.display import display
import qiskit as qk
import qiskit_aer as qka

In [ ]:
def show(qc, title):
    print(title)
    print(qc.draw())
    sv = qk.quantum_info.Statevector.from_instruction(qc)
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()


def show_mpl(qc, title):
    print(title)
    display(qc.draw(output="mpl"))
    sv = qk.quantum_info.Statevector.from_instruction(qc)
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()

## Hadamard creates $|+\rangle$

$H|0\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$. Both amplitudes are
real and positive.

In [ ]:
qc = qk.QuantumCircuit(1)
qc.h(0)
show_mpl(qc, "H|0> = |+>")

## $S$ gate: $\pi/2$ phase

$S|+\rangle = (|0\rangle + i|1\rangle)/\sqrt{2}$. The $|1\rangle$
component gets a factor of $i = e^{i\pi/2}$.

In [ ]:
qc = qk.QuantumCircuit(1)
qc.h(0)
qc.s(0)
show_mpl(qc, "S|+> = (|0> + i|1>)/sqrt(2)")

## $S^\dagger$ undoes $S$

$S^\dagger S|+\rangle = |+\rangle$. The adjoint reverses the phase.

In [ ]:
qc = qk.QuantumCircuit(1)
qc.h(0)
qc.s(0)
qc.sdg(0)
show_mpl(qc, "S+S|+> = |+>")

## $T$ gate: $\pi/4$ phase

$T|+\rangle = (|0\rangle + e^{i\pi/4}|1\rangle)/\sqrt{2}$.
Finer than $S$: only half the phase.

In [ ]:
qc = qk.QuantumCircuit(1)
qc.h(0)
qc.t(0)
show_mpl(qc, "T|+> = (|0> + e^(i*pi/4)|1>)/sqrt(2)")

## Phase gates compose

$T \cdot S|+\rangle$ adds $\pi/2 + \pi/4 = 3\pi/4$ phase.

In [ ]:
qc = qk.QuantumCircuit(1)
qc.h(0)
qc.s(0)
qc.t(0)
show_mpl(qc, "T*S|+> = (|0> + e^(i*3pi/4)|1>)/sqrt(2)")

## Phase is invisible to $Z$-measurement

All phase gates leave $|0\rangle$ and $|1\rangle$ amplitudes equal.
Measuring in the $Z$ basis always gives 50/50.

In [ ]:
circuits = {
    "H|0>": lambda: (lambda qc: (qc.h(0), qc))(qk.QuantumCircuit(1)),
    "S|+>": lambda: (lambda qc: (qc.h(0), qc.s(0), qc))(qk.QuantumCircuit(1)),
    "T|+>": lambda: (lambda qc: (qc.h(0), qc.t(0), qc))(qk.QuantumCircuit(1)),
    "T*S|+>": lambda: (lambda qc: (qc.h(0), qc.s(0), qc.t(0), qc))(qk.QuantumCircuit(1)),
}
backend = qka.AerSimulator()
for name, make in circuits.items():
    qc = make()
    qc.measure_all()
    compiled = qk.transpile(qc, backend)
    counts = backend.run(compiled, shots=2000).result().get_counts()
    print(f"  {name}: {counts}")
print("\nAll ~50/50 — phase is invisible in Z-measurement.")

## Summary

| Gate | Phase on $|1\rangle$ | Matrix |
|------|----------------------|--------|
| $S$ | $e^{i\pi/2} = i$ | $\text{diag}(1, i)$ |
| $T$ | $e^{i\pi/4}$ | $\text{diag}(1, e^{i\pi/4})$ |

Phase gates change *how* $|1\rangle$ interferes, not *how often*
you see it.